In [ ]:
import math
import numpy as np
import plotly.graph_objects as go


In [4]:
def f(x):
    return x**5 - 5*x**3 + 10*x**2 - 5*x
    
def df(x):
    return 5*x**4 - 15*x**2 + 20*x - 5


## Задание 1. 
> выполните решение методом кубической
аппроксимации. Напишите программу, генерирующую последовательность приближений, на
одном из языков программирования, используя 𝜀=0.0001 для критерия останова.

In [ ]:
a, b = -3, -2
epsilon = 0.0001
iteration = 0

In [ ]:

print(f"{'Итер':<5} | {'a':<10} | {'b':<10} | {'x_new':<10} | {'df(x_new)':<10}")
print("-" * 55)

while True:
    iteration += 1
    fa, fb = f(a), f(b)
    dfa, dfb = df(a), df(b)

    # Шаг 1: Вычисление z и w
    z = 3 * (fa - fb) / (b - a) + dfa + dfb
    w = math.sqrt(max(0, z**2 - dfa * dfb))

    # Шаг 2: Вычисление шага theta
    # Кубический полином имеет 2 экстремума, проверяем оба корня
    denom1 = dfa - dfb + 2 * w
    denom2 = dfa - dfb - 2 * w

    theta = None
    if abs(denom1) > 1e-12:
        t1 = (dfa + w - z) / denom1
        if 0 <= t1 <= 1:
            theta = t1

    if theta is None and abs(denom2) > 1e-12:
        t2 = (dfa - w - z) / denom2
        if 0 <= t2 <= 1:
            theta = t2

    if theta is None:
        theta = 0.5 # Резервный шаг для защиты от ошибок округления чисел с плавающей точкой

    # Шаг 3: Новое приближение
    x_new = a + theta * (b - a)
    df_new = df(x_new)

    print(f"{iteration:<5} | {a:<10.6f} | {b:<10.6f} | {x_new:<10.6f} | {df_new:<10.6f}")

    # Шаг 4: Проверка критерия останова
    if abs(df_new) < epsilon:
        print(f"\nРешение найдено!")
        print(f"x* = {x_new:.6f}")
        print(f"f(x*) = {f(x_new):.6f}")
        break

    # Шаг 5: Сужение интервала
    # Сохраняем "вилку" так, чтобы знаки производных на краях всегда были разными
    if dfa * df_new < 0:
        b = x_new
    else:
        a = x_new

Итер  | a          | b          | x_new      | df(x_new) 
-------------------------------------------------------
1     | -3.000000  | -2.000000  | -2.250873  | 2.329656  
2     | -2.250873  | -2.000000  | -2.233887  | -0.018486 
3     | -2.250873  | -2.233887  | -2.234023  | 0.000001  

Решение найдено!
x* = -2.234023
f(x*) = 61.180625


## Задание 2. 
> Реализуйте программно визуализацию исходной и аппроксимирующей
функций в одной координатной плоскости для итерации алгоритма, постройте визуализации
для нескольких итераций.

In [33]:
a, b = -3, -2
epsilon = 0.0001
iteration = 0
max_visualizations = 5

x_wide = np.linspace(-3.5, -1.5, 500)
f_wide = f(x_wide)

In [80]:
def get_p3_coords(a, b):
    fa, fb, dfa, dfb = f(a), f(b), df(a), df(b)
    L = b - a
    t = np.linspace(0, 1, 10_000)
    h00, h10, h01, h11 = 2*t**3-3*t**2+1, t**3-2*t**2+t, -2*t**3+3*t**2, t**3-t**2
    return a + t * L, fa*h00 + dfa*L*h10 + fb*h01 + dfb*L*h11

iterations_data = []
curr_a, curr_b = a, b

print(f"{'Итер':<5} | {'a':<10} | {'b':<10} | {'x_new':<10} | {'df(x_new)':<10} | {'f(x_new)':<10}") 
print("-" * 55)
for i in range(1, max_visualizations):
    fa, fb, dfa, dfb = f(curr_a), f(curr_b), df(curr_a), df(curr_b)
    z = 3 * (fa - fb) / (curr_b - curr_a) + dfa + dfb
    w = np.sqrt(max(0, z**2 - dfa * dfb))
    theta = (dfa + w - z) / (dfa - dfb + 2 * w)
    x_new = curr_a + theta * (curr_b - curr_a)
    df_new = df(x_new)
    
    px, py = get_p3_coords(curr_a, curr_b)
    iterations_data.append({'iter': i, 'x_new': x_new, 'f_new': f(x_new), 'px': px, 'py': py})
    
    print(f"{i:<5} | {curr_a:<10.6f} | {curr_b:<10.6f} | {x_new:<10.6f} | {df_new:<10.6f} | {f(x_new):<10.6f}")
    
    if abs(df_new) < epsilon: 
        print(f"\nРешение найдено!")
        print(f"x* = {x_new:.6f}")
        print(f"f(x*) = {f(x_new):.6f}")
        break
    if dfa * df_new < 0: 
        curr_b = x_new
    else: 
        curr_a = x_new

# --- Визуализация ---
fig = go.Figure()

# 1. Основная функция
x_range = np.linspace(-3, -2, 100_000)
fig.add_trace(go.Scatter(x=x_range, y=f(x_range), name='f(x)', 
                         line=dict(color='rgba(0,0,0,0.3)', width=2, dash='dot')))

# 2. Итерации с подписями
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A']
for i, data in enumerate(iterations_data):
    color = colors[i % len(colors)]
    
    fig.add_trace(go.Scatter(x=data['px'], y=data['py'], name=f'Ит. {data["iter"]}', 
                             line=dict(color=color, width=1.5)))
    
    fig.add_trace(go.Scatter(
        x=[data['x_new']], 
        y=[data['f_new']], 
        mode='markers',
        name=f'x_{data["iter"]}',
        marker=dict(color=color, size=14, symbol='diamond'),
        showlegend=False
    ))

# Настройка осей (как в предыдущем шаге)
fig.update_layout(
    title='Кубическая аппроксимация (Масштаб 1:1)',
    template='plotly_white',
    showlegend=True,
    # Настройка оси X
    xaxis=dict(
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='black',
        showgrid=True,
        gridcolor='lightgray',
        range=[-3.1, -1.9]
    ),
    
    yaxis=dict(
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='black',
        showgrid=True,
        gridcolor='lightgray',
        scaleanchor="x", 
        scaleratio=0.05,     
        range=[55, 65]
    ),

    width=600,   
    height=600
)
fig.show()

Итер  | a          | b          | x_new      | df(x_new)  | f(x_new)  
-------------------------------------------------------
1     | -3.000000  | -2.000000  | -2.250873  | 2.329656   | 61.161105 
2     | -2.250873  | -2.000000  | -2.233887  | -0.018486  | 61.180623 
3     | -2.250873  | -2.233887  | -2.234023  | 0.000001   | 61.180625 

Решение найдено!
x* = -2.234023
f(x*) = 61.180625
